In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!ls -la train_balanced.zip
!file train_balanced.zip
!head -c 300 train_balanced.zip

-rw-r--r-- 1 root root 64100413 Aug 18 06:55 train_balanced.zip
train_balanced.zip: Zip archive data, at least v2.0 to extract, compression method=store
PK     nq]               Fraud/PK     |q]            
����C�����`BB�

In [ ]:
"""
1_train_densenet169.py
=====================
Fine-tunes a torchvision DenseNet169 on the Fraud / Non-Fraud image dataset.
Designed to run top-to-bottom in Google Colab (GPU runtime).

COLAB SETUP
-----------
1. Runtime -> Change runtime type -> GPU (T4 is fine).
2. Upload these 4 files to the Colab session (left sidebar -> Files -> upload):
     - train_balanced.zip
     - val.zip
     - test.zip
     - class_weights.json
3. Run this script top to bottom (as a .py via `!python 1_train_densenet169.py`
   or paste cells into a notebook).

OUTPUT
------
- best_model.pth              (best checkpoint, picked by val F1)
- last_model.pth              (final epoch checkpoint)
- training_history.json       (per-epoch loss/acc/f1 for both splits)
- label_map.json              (class_to_idx mapping used everywhere else)
"""

import os
import json
import time
import zipfile
import copy

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score

# ----------------------------------------------------------------------
# 0. Config
# ----------------------------------------------------------------------
DATA_DIR = "data"                 # unzip target
BATCH_SIZE = 32
NUM_EPOCHS = 15
LR_HEAD = 1e-3                    # lr for the new classifier head
LR_BACKBONE = 1e-4                # lr for unfrozen backbone layers (fine-tune phase)
FREEZE_EPOCHS = 3                 # epochs to train with backbone frozen before unfreezing
IMG_SIZE = 224
NUM_WORKERS = 2
PATIENCE = 5                      # early stopping patience (epochs w/o val F1 improvement)
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
torch.manual_seed(SEED)

# ----------------------------------------------------------------------
# 1. Unzip the uploaded datasets
# ----------------------------------------------------------------------
ZIP_MAP = {
    "train_balanced.zip": os.path.join(DATA_DIR, "train"),
    "val.zip": os.path.join(DATA_DIR, "val"),
    "test.zip": os.path.join(DATA_DIR, "test"),
}

def has_class_subfolders(path):
    """True only if `path` exists AND contains at least one subdirectory
    (i.e. it looks like a real ImageFolder root, not an empty/partial extract)."""
    return os.path.isdir(path) and any(
        entry.is_dir() for entry in os.scandir(path)
    )

for zip_name, target in ZIP_MAP.items():
    if not has_class_subfolders(target):
        assert os.path.exists(zip_name), f"Missing {zip_name} — upload it to Colab first."
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_name, "r") as zf:
            zf.extractall(target)
        if not has_class_subfolders(target):
            raise RuntimeError(
                f"Extracted {zip_name} into {target} but found no class subfolders. "
                f"Contents: {os.listdir(target)[:10]}"
            )
        print(f"Extracted {zip_name} -> {target}")

with open("class_weights.json") as f:
    class_weights_raw = json.load(f)
print("Loaded class weights:", class_weights_raw)

# ----------------------------------------------------------------------
# 2. Transforms & Datasets
# ----------------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(ZIP_MAP["train_balanced.zip"], transform=train_tfms)
val_ds = datasets.ImageFolder(ZIP_MAP["val.zip"], transform=eval_tfms)

print("Class-to-idx mapping:", train_ds.class_to_idx)
assert train_ds.class_to_idx == val_ds.class_to_idx, "Train/val class order mismatch!"

with open("label_map.json", "w") as f:
    json.dump(train_ds.class_to_idx, f, indent=2)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

# Order class_weights tensor by class_to_idx (not dict insertion order)
idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
weight_list = [class_weights_raw[idx_to_class[i]] for i in range(len(idx_to_class))]
class_weights_tensor = torch.tensor(weight_list, dtype=torch.float32).to(device)
print("Class weight tensor (index order):", weight_list, "for classes", idx_to_class)

# ----------------------------------------------------------------------
# 3. Model — DenseNet169 pretrained on ImageNet, new binary head
# ----------------------------------------------------------------------
model = models.densenet169(weights=models.DenseNet169_Weights.IMAGENET1K_V1)

# Freeze the backbone initially
for param in model.parameters():
    param.requires_grad = False

num_classes = len(train_ds.classes)
# DenseNet's classifier head is `model.classifier` (a single Linear layer),
# not `model.fc` like ResNet.
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier.in_features, num_classes),
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LR_HEAD)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
                                                         factor=0.5, patience=2)

# ----------------------------------------------------------------------
# 4. Train / Eval loops
# ----------------------------------------------------------------------
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1


history = {"train_loss": [], "train_acc": [], "train_f1": [],
           "val_loss": [], "val_acc": [], "val_f1": []}

best_f1 = -1.0
best_state = None
epochs_no_improve = 0
unfrozen = False

print("\n=== Phase 1: training classifier head (backbone frozen) ===")
for epoch in range(NUM_EPOCHS):
    # Unfreeze backbone after FREEZE_EPOCHS, add its params with a lower LR
    if epoch == FREEZE_EPOCHS and not unfrozen:
        print("\n=== Phase 2: unfreezing backbone for fine-tuning ===")
        for param in model.parameters():
            param.requires_grad = True
        optimizer = torch.optim.Adam([
            {"params": model.classifier.parameters(), "lr": LR_HEAD},
            {"params": [p for n, p in model.named_parameters() if not n.startswith("classifier")],
             "lr": LR_BACKBONE},
        ])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
                                                                 factor=0.5, patience=2)
        unfrozen = True

    t0 = time.time()
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_f1 = run_epoch(val_loader, train=False)
    scheduler.step(val_f1)
    dt = time.time() - t0

    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc); history["train_f1"].append(tr_f1)
    history["val_loss"].append(val_loss); history["val_acc"].append(val_acc); history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} ({dt:.1f}s) | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} f1 {tr_f1:.4f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        torch.save(best_state, "best_model.pth")
        print(f"  -> new best model saved (val F1 {best_f1:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping — no val F1 improvement in {PATIENCE} epochs.")
            break

torch.save(model.state_dict(), "last_model.pth")
with open("training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\nDone. Best val macro-F1: {best_f1:.4f}")
print("Saved: best_model.pth, last_model.pth, training_history.json, label_map.json")

Using device: cuda
Loaded class weights: {'Fraud': 2.017857142857143, 'Non-Fraud': 0.6647058823529411}
Class-to-idx mapping: {'Fraud': 0, 'Non-Fraud': 1}
Class weight tensor (index order): [2.017857142857143, 0.6647058823529411] for classes {0: 'Fraud', 1: 'Non-Fraud'}

=== Phase 1: training classifier head (backbone frozen) ===
Epoch 01/15 (29.6s) | train loss 0.4152 acc 0.8136 f1 0.7728 | val loss 0.2256 acc 0.9210 f1 0.6770
  -> new best model saved (val F1 0.6770)
Epoch 02/15 (27.5s) | train loss 0.2972 acc 0.8828 f1 0.8536 | val loss 0.2370 acc 0.9123 f1 0.6923
  -> new best model saved (val F1 0.6923)
Epoch 03/15 (29.8s) | train loss 0.2710 acc 0.8913 f1 0.8632 | val loss 0.1933 acc 0.9321 f1 0.6863

=== Phase 2: unfreezing backbone for fine-tuning ===
Epoch 04/15 (76.2s) | train loss 0.1406 acc 0.9519 f1 0.9368 | val loss 0.1399 acc 0.9691 f1 0.8655
  -> new best model saved (val F1 0.8655)
Epoch 05/15 (75.7s) | train loss 0.0406 acc 0.9867 f1 0.9823 | val loss 0.2593 acc 0.9593

In [ ]:
"""
2_evaluate_and_visualize.py
============================
Loads the checkpoint produced by 1_train_densenet169.py, runs it on the held-out
test set, and produces a full performance report + visualizations.

Run this AFTER 1_train_densenet169.py, in the same Colab session (or after
re-uploading best_model.pth / label_map.json / test.zip to a fresh session).

OUTPUT (all saved into ./eval_outputs/)
----------------------------------------
- confusion_matrix.png
- roc_curve.png
- precision_recall_curve.png
- training_curves.png              (if training_history.json is present)
- metrics_report.json              (accuracy, precision, recall, F1, AUC, per-class)
- misclassified_grid.png           (sample of wrong predictions, for a gut-check)
"""

import os
import json
import zipfile

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# ----------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------
DATA_DIR = "data"
IMG_SIZE = 224
BATCH_SIZE = 32
OUT_DIR = "eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ----------------------------------------------------------------------
# Unzip test set if needed
# ----------------------------------------------------------------------
import glob

def find_zip(keyword):
    """Find a zip file in the current directory whose name contains `keyword`,
    regardless of prefixes/suffixes Colab's upload may have added."""
    matches = [f for f in glob.glob("*.zip") if keyword.lower() in f.lower()]
    if not matches:
        raise FileNotFoundError(
            f"No zip file found containing '{keyword}' in the current directory. "
            f"Files present: {glob.glob('*.zip')}. Upload it to Colab first."
        )
    if len(matches) > 1:
        print(f"Warning: multiple zips matched '{keyword}': {matches} — using {matches[0]}")
    return matches[0]

def has_class_subfolders(path):
    """True only if `path` exists AND contains at least one subdirectory
    (i.e. it looks like a real ImageFolder root, not an empty/partial extract)."""
    return os.path.isdir(path) and any(
        entry.is_dir() for entry in os.scandir(path)
    )

test_dir = os.path.join(DATA_DIR, "test")
if not has_class_subfolders(test_dir):
    test_zip = find_zip("test")
    os.makedirs(test_dir, exist_ok=True)
    with zipfile.ZipFile(test_zip, "r") as zf:
        zf.extractall(test_dir)
    if not has_class_subfolders(test_dir):
        raise RuntimeError(
            f"Extracted {test_zip} into {test_dir} but found no class subfolders. "
            f"Contents: {os.listdir(test_dir)[:10]}"
        )

with open("label_map.json") as f:
    class_to_idx = json.load(f)
idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
print("Classes:", class_names)

# ----------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_ds = datasets.ImageFolder(test_dir, transform=eval_tfms)
assert test_ds.class_to_idx == class_to_idx, "Test set class order doesn't match training label_map.json!"
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ----------------------------------------------------------------------
# Model — DenseNet169
# ----------------------------------------------------------------------
model = models.densenet169(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier.in_features, len(class_names)),
)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model = model.to(device)
model.eval()

# ----------------------------------------------------------------------
# Run inference on the test set
# ----------------------------------------------------------------------
all_labels, all_preds, all_probs, all_paths = [], [], [], []
softmax = nn.Softmax(dim=1)

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = softmax(outputs).cpu().numpy()
        preds = probs.argmax(axis=1)

        all_labels.extend(labels.numpy().tolist())
        all_preds.extend(preds.tolist())
        all_probs.extend(probs.tolist())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_paths = [p for p, _ in test_ds.samples]

# The "positive" class for ROC/PR — assume "Fraud" is the class of interest
fraud_idx = class_to_idx.get("Fraud", 1)
fraud_probs = all_probs[:, fraud_idx]
fraud_true = (all_labels == fraud_idx).astype(int)

# ----------------------------------------------------------------------
# 1. Metrics report
# ----------------------------------------------------------------------
report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
fpr, tpr, _ = roc_curve(fraud_true, fraud_probs)
roc_auc = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(fraud_true, fraud_probs)
pr_auc = average_precision_score(fraud_true, fraud_probs)

metrics_summary = {
    "overall_accuracy": report["accuracy"],
    "macro_f1": report["macro avg"]["f1-score"],
    "weighted_f1": report["weighted avg"]["f1-score"],
    "roc_auc_fraud": roc_auc,
    "pr_auc_fraud": pr_auc,
    "per_class": {cls: report[cls] for cls in class_names},
    "n_test_samples": len(all_labels),
}
with open(os.path.join(OUT_DIR, "metrics_report.json"), "w") as f:
    json.dump(metrics_summary, f, indent=2)

print("\n=== TEST SET PERFORMANCE ===")
print(f"Accuracy:      {metrics_summary['overall_accuracy']:.4f}")
print(f"Macro F1:      {metrics_summary['macro_f1']:.4f}")
print(f"ROC-AUC (Fraud): {roc_auc:.4f}")
print(f"PR-AUC  (Fraud): {pr_auc:.4f}")
for cls in class_names:
    p = report[cls]
    print(f"  {cls:10s} precision {p['precision']:.3f}  recall {p['recall']:.3f}  f1 {p['f1-score']:.3f}")

# ----------------------------------------------------------------------
# 2. Confusion matrix
# ----------------------------------------------------------------------
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names)
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Test Set")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------
# 3. ROC curve
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})", linewidth=2)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random chance")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Fraud Detection")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "roc_curve.png"), dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------
# 4. Precision-Recall curve (more informative than ROC for imbalanced data)
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(recall, precision, label=f"PR curve (AP = {pr_auc:.3f})", linewidth=2, color="darkorange")
baseline = fraud_true.sum() / len(fraud_true)
ax.axhline(baseline, linestyle="--", color="gray", label=f"Baseline ({baseline:.2f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — Fraud Detection")
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "precision_recall_curve.png"), dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------
# 5. Training curves (if history exists)
# ----------------------------------------------------------------------
if os.path.exists("training_history.json"):
    with open("training_history.json") as f:
        hist = json.load(f)
    epochs = range(1, len(hist["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(epochs, hist["train_loss"], label="train")
    axes[0].plot(epochs, hist["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(epochs, hist["train_acc"], label="train")
    axes[1].plot(epochs, hist["val_acc"], label="val")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

    axes[2].plot(epochs, hist["train_f1"], label="train")
    axes[2].plot(epochs, hist["val_f1"], label="val")
    axes[2].set_title("Macro F1"); axes[2].set_xlabel("Epoch"); axes[2].legend()

    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150)
    plt.close(fig)

# ----------------------------------------------------------------------
# 6. Grid of misclassified examples (quick visual gut-check)
# ----------------------------------------------------------------------
from PIL import Image

wrong_idx = np.where(all_preds != all_labels)[0]
n_show = min(12, len(wrong_idx))
if n_show > 0:
    sample_idx = np.random.choice(wrong_idx, n_show, replace=False)
    cols = 4
    rows = int(np.ceil(n_show / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).reshape(-1)
    for ax, idx in zip(axes, sample_idx):
        img = Image.open(all_paths[idx]).convert("RGB")
        ax.imshow(img)
        true_c = idx_to_class[all_labels[idx]]
        pred_c = idx_to_class[all_preds[idx]]
        conf = all_probs[idx][all_preds[idx]]
        ax.set_title(f"true: {true_c}\npred: {pred_c} ({conf:.2f})", fontsize=9, color="crimson")
        ax.axis("off")
    for ax in axes[n_show:]:
        ax.axis("off")
    fig.suptitle("Misclassified Test Samples", fontsize=13)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "misclassified_grid.png"), dpi=150)
    plt.close(fig)
    print(f"\nSaved {n_show} misclassified examples to misclassified_grid.png")
else:
    print("\nNo misclassified examples — perfect test accuracy (double-check for leakage!).")

print(f"\nAll visualizations + metrics_report.json saved to ./{OUT_DIR}/")

Using device: cuda
Classes: ['Fraud', 'Non-Fraud']

=== TEST SET PERFORMANCE ===
Accuracy:      0.9654
Macro F1:      0.8745
ROC-AUC (Fraud): 0.9798
PR-AUC  (Fraud): 0.8888
  Fraud      precision 0.686  recall 0.871  f1 0.768
  Non-Fraud  precision 0.991  recall 0.972  f1 0.981

Saved 12 misclassified examples to misclassified_grid.png

All visualizations + metrics_report.json saved to ./eval_outputs/


In [ ]:
"""
3_dashboard_app.py
====================
A polished Gradio dashboard for the Fraud / Non-Fraud DenseNet169 classifier.

Lets you:
  - Upload one image and get a prediction + confidence bar chart + Grad-CAM
    heatmap (shows WHERE the model is looking).
  - See the model's overall test-set performance metrics on the same screen
    (reads eval_outputs/metrics_report.json from step 2, if present).

RUN IN COLAB
------------
1. Make sure best_model.pth and label_map.json are in the working directory
   (produced by 1_train_densenet169.py).
2. (Optional) Run 2_evaluate_and_visualize.py first so the "Model Performance"
   tab has real numbers to show — otherwise that tab just says "not found".
3. !pip install gradio grad-cam -q
4. !python 3_dashboard_app.py
   Colab will print a public gradio.live link — click it to open the dashboard.

RUN LOCALLY
-----------
   pip install gradio grad-cam torch torchvision pillow
   python 3_dashboard_app.py
   -> opens http://127.0.0.1:7860
"""

import os
import json

import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import gradio as gr

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# ----------------------------------------------------------------------
# Load model + label map
# ----------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("label_map.json") as f:
    class_to_idx = json.load(f)
idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

model = models.densenet169(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier.in_features, len(class_names)),
)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model = model.to(device)
model.eval()

# Grad-CAM targets the final batch-norm layer of DenseNet169's feature
# extractor (the last spatial layer before global pooling / classifier).
cam = GradCAM(model=model, target_layers=[model.features.norm5])

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ----------------------------------------------------------------------
# Load pre-computed test-set metrics, if available
# ----------------------------------------------------------------------
METRICS_PATH = "eval_outputs/metrics_report.json"
metrics_summary = None
if os.path.exists(METRICS_PATH):
    with open(METRICS_PATH) as f:
        metrics_summary = json.load(f)

# ----------------------------------------------------------------------
# Prediction + Grad-CAM
# ----------------------------------------------------------------------
def predict(image: Image.Image):
    if image is None:
        return None, None, "Upload an image first."

    image = image.convert("RGB")
    img_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    pred_class = idx_to_class[pred_idx]
    confidence = float(probs[pred_idx])

    label_dict = {class_names[i]: float(probs[i]) for i in range(len(class_names))}

    # Grad-CAM heatmap overlaid on the resized original image
    rgb_img = np.array(image.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
    grayscale_cam = cam(input_tensor=img_tensor, targets=None)[0]
    cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    verdict = f"### Prediction: **{pred_class}**  ({confidence*100:.1f}% confidence)"
    return label_dict, cam_image, verdict


# ----------------------------------------------------------------------
# Metrics tab content
# ----------------------------------------------------------------------
def metrics_markdown():
    if metrics_summary is None:
        return ("No evaluation results found yet. Run `2_evaluate_and_visualize.py` "
                "first, then restart this app to populate this tab.")

    lines = [
        f"## Test Set Performance ({metrics_summary['n_test_samples']} images)",
        "",
        f"| Metric | Value |",
        f"|---|---|",
        f"| Accuracy | {metrics_summary['overall_accuracy']:.3f} |",
        f"| Macro F1 | {metrics_summary['macro_f1']:.3f} |",
        f"| Weighted F1 | {metrics_summary['weighted_f1']:.3f} |",
        f"| ROC-AUC (Fraud) | {metrics_summary['roc_auc_fraud']:.3f} |",
        f"| PR-AUC (Fraud) | {metrics_summary['pr_auc_fraud']:.3f} |",
        "",
        "### Per-class breakdown",
        "| Class | Precision | Recall | F1 |",
        "|---|---|---|---|",
    ]
    for cls, vals in metrics_summary["per_class"].items():
        lines.append(f"| {cls} | {vals['precision']:.3f} | {vals['recall']:.3f} | {vals['f1-score']:.3f} |")
    return "\n".join(lines)


EVAL_IMAGES = {
    "Confusion Matrix": "eval_outputs/confusion_matrix.png",
    "ROC Curve": "eval_outputs/roc_curve.png",
    "Precision-Recall Curve": "eval_outputs/precision_recall_curve.png",
    "Training Curves": "eval_outputs/training_curves.png",
}
available_eval_images = [(name, path) for name, path in EVAL_IMAGES.items() if os.path.exists(path)]

# ----------------------------------------------------------------------
# Build the Gradio UI
# ----------------------------------------------------------------------
theme = gr.themes.Soft(
    primary_hue="rose",
    secondary_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
).set(
    body_background_fill="*neutral_50",
    block_shadow="*shadow_drop_lg",
    block_radius="16px",
)

CUSTOM_CSS = """
#title-block {text-align: center; margin-bottom: 0.5rem;}
#title-block h1 {font-weight: 700; letter-spacing: -0.5px;}
#verdict-box {font-size: 1.2rem; padding: 0.5rem;}
"""

with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="Fraud Image Classifier") as demo:
    with gr.Column(elem_id="title-block"):
        gr.Markdown("# 🕵️ Fraud vs Non-Fraud Image Classifier")
        gr.Markdown("DenseNet169 (transfer-learned) — upload an image to classify it and see what the model focused on.")

    with gr.Tabs():
        with gr.Tab("🔍 Classify an Image"):
            with gr.Row():
                with gr.Column(scale=1):
                    img_input = gr.Image(type="pil", label="Upload image", height=320)
                    predict_btn = gr.Button("Classify", variant="primary")
                with gr.Column(scale=1):
                    verdict_out = gr.Markdown(elem_id="verdict-box")
                    label_out = gr.Label(label="Confidence by class", num_top_classes=2)
                    cam_out = gr.Image(label="Grad-CAM — where the model is looking", height=320)

            predict_btn.click(fn=predict, inputs=img_input, outputs=[label_out, cam_out, verdict_out])
            img_input.change(fn=predict, inputs=img_input, outputs=[label_out, cam_out, verdict_out])

        with gr.Tab("📊 Model Performance"):
            gr.Markdown(metrics_markdown())
            if available_eval_images:
                with gr.Row():
                    for name, path in available_eval_images:
                        with gr.Column():
                            gr.Markdown(f"**{name}**")
                            gr.Image(value=path, show_label=False)

if __name__ == "__main__":
    demo.launch(share=True)

/tmp/ipykernel_370/2202212863.py:168: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="Fraud Image Classifier") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://85a4adcb497799e189.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.path.isdir("/content/drive/MyDrive"))

True


In [ ]:
# ----------------------------------------------------------------------
# Save trained model + artifacts to Google Drive
# ----------------------------------------------------------------------
# This cell is self-contained: it mounts Drive itself if needed, and
# defines SAVE_DIR itself, so it doesn't depend on any earlier cell
# still being in this kernel's memory (e.g. after a runtime restart).

import os
import shutil
import datetime
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

assert os.path.isdir("/content/drive/MyDrive"), (
    "Drive mount failed. Try drive.mount('/content/drive', force_remount=True)."
)

SAVE_DIR = "/content/drive/MyDrive/fraud_classifier/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# Files produced by the training / eval scripts, if present
FILES_TO_SAVE = [
    "best_model.pth",
    "last_model.pth",
    "training_history.json",
    "label_map.json",
]

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = os.path.join(SAVE_DIR, f"run_{timestamp}")
os.makedirs(run_dir, exist_ok=True)

saved, missing = [], []
for fname in FILES_TO_SAVE:
    if os.path.exists(fname):
        # Copy into a timestamped run folder (keeps history of every run)...
        shutil.copy(fname, os.path.join(run_dir, fname))
        # ...and also into the top-level SAVE_DIR (always points at latest run)
        shutil.copy(fname, os.path.join(SAVE_DIR, fname))
        saved.append(fname)
    else:
        missing.append(fname)

# Optionally include the eval_outputs/ folder (metrics + plots) if it exists
if os.path.isdir("eval_outputs"):
    dst = os.path.join(run_dir, "eval_outputs")
    shutil.copytree("eval_outputs", dst, dirs_exist_ok=True)
    shutil.copytree("eval_outputs", os.path.join(SAVE_DIR, "eval_outputs"), dirs_exist_ok=True)
    saved.append("eval_outputs/")

print(f"Saved to Drive: {run_dir}")
print(f"Also mirrored to latest-run copy at: {SAVE_DIR}")
print(f"Files saved: {saved}")
if missing:
    print(f"Not found locally (skipped): {missing}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Drive: /content/drive/MyDrive/fraud_classifier/checkpoints/run_20260818_073116
Also mirrored to latest-run copy at: /content/drive/MyDrive/fraud_classifier/checkpoints
Files saved: ['best_model.pth', 'last_model.pth', 'training_history.json', 'label_map.json', 'eval_outputs/']
